# v0.2.8 live smoke test, run 3 (final pre-freeze check): EMU_Deccan_forest + the selected aquatic zone + parity

Reruns ONLY: `EMU_Deccan_forest`, the selected valid Tata aquatic zone, and the parity harness. No other Tata zone, no new indicator, no
headline-aggregation change. `main` is untouched.

Two fixes since run 2: (1) **aquatic permanence** is now one definition for every aquatic indicator (mean Sentinel-1 occurrence over the
water body's own pixels, 10 m grid); run 2 measured it over the land ring for `sdi` / `riparian_natural_veg_share` (0.031 vs 0.91), which
silently switched the permanence criterion off for them; (2) **provenance**: every output records the commit, branch, dirty flag, a source
fingerprint, package / contract / code version and the full configuration, captured when the code was *imported*, and flags a checkout that
changed afterwards.

**Restart the runtime first (Runtime -> Restart session), then run the cells in order.** Cell 4b stops the run if the loaded code is not the
checkout on disk.

In [ ]:
# 1. Clone / update the v0.2.8 BRANCH (main is untouched). Safe to re-run.
import os, glob
REPO_DIR = '/content/reference-benchmarking'
BRANCH = 'v0.2.8-contract'
if not os.path.exists(REPO_DIR):
    !git clone --branch $BRANCH https://github.com/G-auravSingh/reference-benchmarking.git $REPO_DIR
else:
    !git -C $REPO_DIR fetch origin $BRANCH
    !git -C $REPO_DIR checkout $BRANCH
    !git -C $REPO_DIR pull --ff-only origin $BRANCH
PKG_DIR = sorted(glob.glob(f'{REPO_DIR}/darukaa_reference_v*'))[-1]
!git -C $REPO_DIR log --oneline -1
%cd $PKG_DIR
!pip install -q -r requirements.txt
!pip install -q -e .
!pip install -q pyproj shapely

In [ ]:
# 2. Earth Engine (same project as the main notebook)
import ee
ee.Authenticate()
ee.Initialize(project='gaurav-singh-007')
print('Earth Engine ready')

In [ ]:
# 3. Config: identical to the main notebook's Tata Motors config
from darukaa_reference.config import Config
from darukaa_reference.indicators import create_default_registry
config = Config(
    gee_project='gaurav-singh-007', output_dir='outputs', archetype='industrial',
    realm='terrestrial', assessment_mode='baseline',
    hmi_hard_ceiling=0.05, use_variance_stability_floor=True, reference_stratification='ecoregion_landcover',
)
registry = create_default_registry()
print('riparian ring width (m):', config.riparian_ring_width_m, '| early period:', config.net_change_early_years)

In [ ]:
# 3b. PROVENANCE GATE: what code did this process actually load? Stops if the checkout changed after import (stale kernel) or the tree is dirty.
from darukaa_reference import provenance as PV
prov = PV.run_provenance(config)
print(PV.brief(prov))
print('loaded commit:', prov['git_commit'], '| branch:', prov['branch'], '| on disk now:', prov['checkout_now'])
!git -C $REPO_DIR log --oneline -1
assert not prov['warnings'], 'PROVENANCE WARNING: ' + '; '.join(prov['warnings'])

In [ ]:
# 4. Locate the two zones from the Tata manifests
import sys, os
sys.path.insert(0, os.getcwd())
from run_project_from_manifest import find_manifest_by_project_name, load_manifest, resolve_tile_paths
REPO_ROOT = os.path.dirname(os.getcwd())
PATHS = {}
for proj in ('TataMotors_Pimpri', 'TataMotors_Pimpri_Aquatic'):
    mp = find_manifest_by_project_name(REPO_ROOT, proj)
    man = load_manifest(mp)
    PATHS.update(dict(zip(man['tile_labels'], resolve_tile_paths(man, mp))))
TERRESTRIAL_LABEL, AQUATIC_LABEL = 'EMU_Deccan_forest', 'Lake_Suman'
print(TERRESTRIAL_LABEL, '->', PATHS[TERRESTRIAL_LABEL]); print(AQUATIC_LABEL, '->', PATHS[AQUATIC_LABEL])

## Step A. Aquatic zone selection (evidence only, cheap)
Measures the water body the aquatic indicators would use: water-mask area inside the site, the target body's pixel-count area, bounding
box and **pure-water pixel count**, and the ecosystem classification. `Lake_Suman` is used if it satisfies the open-water definition
(a complete water body with >= 10 pure-water pixels); otherwise the next existing Tata aquatic tile that does is used. A probe that
*fails* is shown as an error, never as "no water".

In [ ]:
# 5. Evidence for the aquatic candidates (evidence only: no reference building). Nothing here is invented: candidates are the existing Tata aquatic tiles.
import json, logging
logging.basicConfig(level=logging.WARNING)
from darukaa_reference import assess as A
CANDIDATES = ['Lake_Suman', 'Lake_Sharma', 'Pond_1', 'Pond_2', 'Pond_3', 'Pond_4']
CANDIDATES = [c for c in CANDIDATES if c in PATHS] or [k for k in PATHS if 'Lake' in k or 'Pond' in k]
selected, evidence_log = None, {}
for lab in CANDIDATES:
    z = A.load_zone(config, registry, PATHS[lab], lab, 'aquatic')
    prov = A.EEProvider(config, registry)
    ev = prov.evidence(z)
    evidence_log[lab] = {'water_probe': ev.water_probe, 'water_probe_detail': ev.water_probe_detail, 'water_body_valid': ev.water_body_valid,
                         'n_pure_water_px': ev.n_pure_water_px, 'invalid_reason': ev.water_body_invalid_reason,
                         'ecosystem_tags': sorted(ev.ecosystem_tags), 'site_area_m2': ev.site_area_m2, 'diagnostics': ev.diagnostics}
    d = ev.diagnostics
    print(f"\n=== {lab}: site {ev.site_area_m2/1e4:.2f} ha | probe={ev.water_probe} {ev.water_probe_detail} | valid={ev.water_body_valid} | pure px={ev.n_pure_water_px} | classification={d.get('ecosystem_classification')}")
    print('    water mask:', json.dumps(d.get('water_mask'), default=str)[:400])
    print('    body probe:', json.dumps(d.get('water_body_probe'), default=str)[:700])
    if d.get('errors'): print('    ERRORS:', d['errors'])
    if selected is None and ev.water_probe == 'ok' and ev.water_body_valid and (ev.n_pure_water_px or 0) >= 10:
        selected = lab
        break                                    # first zone that satisfies the definition: stop, do not touch more tiles
AQUATIC_LABEL = selected
print('\nSELECTED AQUATIC ZONE:', AQUATIC_LABEL)
assert AQUATIC_LABEL, 'no existing Tata aquatic tile satisfies the open-water definition: STOP and report the table above'

## Step B. Smoke test A: terrestrial zone

In [ ]:
# 6. SMOKE TEST A: EMU_Deccan_forest (writes outputs/smoke/EMU_Deccan_forest_audit.{csv,json,md}; the JSON has meta.evidence)
import time
TERRESTRIAL_LABEL = 'EMU_Deccan_forest'
zoneA = A.load_zone(config, registry, PATHS[TERRESTRIAL_LABEL], TERRESTRIAL_LABEL, 'terrestrial')
print('ecoregion id:', zoneA.eco_id)
providerA = A.EEProvider(config, registry)
rowsA = A.run_smoke_test(config, registry, zoneA, out_dir='outputs/smoke', provider=providerA)

## Step C. Smoke test B: the selected aquatic zone

In [ ]:
# 7. SMOKE TEST B: the aquatic zone selected in Step A
zoneB = A.load_zone(config, registry, PATHS[AQUATIC_LABEL], AQUATIC_LABEL, 'aquatic')
print('ecoregion id:', zoneB.eco_id)
providerB = A.EEProvider(config, registry)
rowsB = A.run_smoke_test(config, registry, zoneB, out_dir='outputs/smoke', provider=providerB)

## Step D. Methodological validation zone (forest loss), parity only
Run 2 selected `FCF_GV_FCF_GV_EMU_Ganjam_1` (>= 5 ha baseline, non-zero loss). It is reused here for the forest **parity** checks only; it is
a methodological validation dataset, never in Tata scoring. Its forest-loss *reference* returned 0 valid cells in run 2 (43 eligible by the
population mask, all masked by the metric): that is an open finding, not re-run here (set `RUN_FOREST_ASSESSMENT = True` to re-run it with the new
shortfall diagnostics).

In [ ]:
# 8. Reuse the run-2 validation zone (no re-selection). Assessment is OFF by default: only the parity checks use it.
RUN_FOREST_ASSESSMENT = False
VALIDATION_LABEL = 'FCF_GV_FCF_GV_EMU_Ganjam_1'
cands = A.find_candidate_tiles(REPO_ROOT, top=12)
c = next((c for c in cands if c['label'] == VALIDATION_LABEL), None)
assert c, f'{VALIDATION_LABEL} not found among existing assets: ' + str([x["label"] for x in cands])
import dataclasses
zoneV = dataclasses.replace(A.load_zone(config, registry, c['path'], c['label'], 'terrestrial'), validation_label=A.FOREST_VALIDATION_LABEL)
print('VALIDATION ZONE:', zoneV.label, '|', zoneV.validation_label, '|', round(c['area_ha'], 1), 'ha')
if RUN_FOREST_ASSESSMENT:
    rowsV = A.run_smoke_test(config, registry, zoneV, out_dir='outputs/smoke', provider=providerA, only=['forest_loss_rate'])

## Step E. Parity
`DISCREPANCY` = a real finding; `HARNESS_ERROR` = the check itself failed (not evidence about the indicators); `INFO` = quantified, not judged.
Includes the **live synthetic water-body fixture** and, if a validation zone was found, the Hansen bundle and site-rate parity on it.

In [ ]:
# 9. EE-vs-numpy parity (each check isolated; masked bands reported, not fatal)
from darukaa_reference import parity
results = parity.run_parity(config, registry, zoneA, zoneB, out_dir='outputs/smoke', zone_forest=(zoneV if 'zoneV' in globals() else None))

In [ ]:
# 10. Provenance, aquatic permanence consistency, and what the requests cost
import json, glob
lines = []
for f in sorted(glob.glob('outputs/smoke/*_audit.json')):
    d = json.load(open(f)); pv = d.get('provenance', {}); m = d.get('meta', {})
    lines.append(f"== {f}\nprovenance: commit {pv.get('git_commit_short')} branch {pv.get('branch')} dirty={pv.get('dirty')} contract {pv.get('contract_version')} code '{pv.get('code_version')}' config {str(pv.get('config_sha256'))[:10]} warnings={pv.get('warnings')}")
    ap = m.get('aquatic_permanence')
    if ap: lines.append(f"aquatic target permanence by indicator: {ap['target_permanence_by_indicator']} | spread {ap['spread']:.4f} | consistent={ap['consistent']}")
    e = m.get('evidence', {}); lines.append(f"evidence: probe={e.get('water_probe')} valid={e.get('water_body_valid')} pure_px={e.get('n_pure_water_px')}")
    for r in d['rows']:
        if r['seconds'] > 30 or r['reason'].startswith('provider_error') or r['reason'] == 'applicability_undetermined':
            lines.append(f"  {r['indicator']:30s} {r['status']:32s} {r['reason'][:50]:50s} {r['seconds']:8.1f}s n={r['reference_n']}")
open('outputs/smoke/provenance_and_tiling.txt', 'w').write('\n'.join(lines)); print('\n'.join(lines))

In [ ]:
# 11. Zip everything to send back
import shutil
shutil.make_archive('/content/v028_smoke_results_run3', 'zip', 'outputs/smoke')
from google.colab import files
files.download('/content/v028_smoke_results_run3.zip')